# Inventory Replenishment Agent
This is the submission-ready notebook. It loads the required CSV files, uses an EWMA forecast, computes safety stock from forecast error and service level, makes daily order-or-wait decisions, simulates purchase-order arrivals after lead time, and compares the agent against a simple fixed-rule baseline.

## Checklist coverage
- Loads `sales.csv`, `inventory.csv`, and `params.csv`
- Aggregates daily demand per SKU
- Uses EWMA forecasting that updates daily
- Computes safety stock from forecast error + service level
- Respects `min_order_qty` and lead time
- Simulates PO receipts, stockouts, fill rate, and total cost
- Prints daily rationale logs and compares against a baseline

In [1]:
from pathlib import Path
import pandas as pd

BASE = Path('.')
sales = pd.read_csv(BASE / 'sales.csv', parse_dates=['date'])
inventory = pd.read_csv(BASE / 'inventory.csv')
params = pd.read_csv(BASE / 'params.csv')

sales_daily = sales.groupby(['date', 'sku'], as_index=False)['qty_sold'].sum().sort_values(['date', 'sku'])
print('Sales preview:')
display(sales_daily.head(9))
print('Inventory preview:')
display(inventory)
print('Params preview:')
display(params)

Sales preview:


,date,sku,qty_sold
0,2024-01-01,SKU_A,19
1,2024-01-01,SKU_B,9
2,2024-01-01,SKU_C,8
3,2024-01-02,SKU_A,23
4,2024-01-02,SKU_B,10
5,2024-01-02,SKU_C,7
6,2024-01-03,SKU_A,22
7,2024-01-03,SKU_B,13
8,2024-01-03,SKU_C,9


Inventory preview:


,sku,opening_stock
0,SKU_A,180
1,SKU_B,120
2,SKU_C,90


Params preview:


,sku,unit_cost,holding_cost_per_day,stockout_cost,lead_time_days,min_order_qty,service_level
0,SKU_A,12.0,0.08,6.0,5,30,0.95
1,SKU_B,7.0,0.05,5.0,4,20,0.93
2,SKU_C,4.0,0.03,4.0,3,15,0.90


In [2]:
from inventory_replenishment_agent_fixed import load_inputs, simulate_policy

sales_df, inventory_df, params_df = load_inputs('sales.csv', 'inventory.csv', 'params.csv')
agent_logs, agent_per_sku, agent_summary = simulate_policy(sales_df, inventory_df, params_df, policy_name='agent', alpha=0.30, review_period_days=1)
baseline_logs, baseline_per_sku, baseline_summary = simulate_policy(sales_df, inventory_df, params_df, policy_name='baseline', alpha=0.30, review_period_days=1)
comparison = pd.concat([agent_summary, baseline_summary], ignore_index=True)
comparison

,policy,stockout_units,stockout_days,fill_rate,holding_cost,stockout_cost,total_cost
0,agent,12.0,5,0.9965,407.54,53.0,460.54
1,baseline,371.0,45,0.8927,532.35,1964.0,2496.35


## Interpretation
The agent uses a daily-updating EWMA forecast and safety stock, so it reacts to demand changes more intelligently than the fixed-rule baseline. The baseline is simpler, but it is less responsive to changing demand and can create more stockouts.

In [3]:
print('Per-SKU KPIs for the agent:')
display(agent_per_sku)

print('Per-SKU KPIs for the baseline:')
display(baseline_per_sku)

Per-SKU KPIs for the agent:


,policy,sku,total_demand,filled_units,stockout_units,stockout_days,fill_rate,holding_cost,stockout_cost,total_cost
0,agent,SKU_A,1737.0,1737.0,0.0,0,1.0000,258.24,0.0,258.24
1,agent,SKU_B,1084.0,1079.0,5.0,2,0.9954,102.80,25.0,127.80
2,agent,SKU_C,636.0,629.0,7.0,3,0.9890,46.50,28.0,74.50


Per-SKU KPIs for the baseline:


,policy,sku,total_demand,filled_units,stockout_units,stockout_days,fill_rate,holding_cost,stockout_cost,total_cost
0,baseline,SKU_A,1737.0,1559.0,178.0,13,0.8975,374.16,1068.0,1442.16
1,baseline,SKU_B,1084.0,960.0,124.0,16,0.8856,117.30,620.0,737.30
2,baseline,SKU_C,636.0,567.0,69.0,16,0.8915,40.89,276.0,316.89


In [4]:
print('Agent daily log sample:')
display(agent_logs[['date','sku','forecast','safety_stock','inventory_position','target_stock','action','order_qty','actual_demand','stockout_units','rationale']].head(12))

print('Baseline daily log sample:')
display(baseline_logs[['date','sku','forecast','safety_stock','inventory_position','target_stock','action','order_qty','actual_demand','stockout_units','rationale']].head(12))

Agent daily log sample:


,date,sku,forecast,safety_stock,inventory_position,target_stock,action,order_qty,actual_demand,stockout_units,rationale
0,2024-01-01,SKU_A,19.00,19.14,180.0,133.14,wait,0,19.0,0.0,Projected inventory (180.0) is above target st...
1,2024-01-01,SKU_B,9.00,7.42,120.0,52.42,wait,0,9.0,0.0,Projected inventory (120.0) is above target st...
2,2024-01-01,SKU_C,8.00,5.13,90.0,37.13,wait,0,8.0,0.0,Projected inventory (90.0) is above target sto...
3,2024-01-02,SKU_A,19.00,19.14,161.0,133.14,wait,0,23.0,0.0,Projected inventory (161.0) is above target st...
4,2024-01-02,SKU_B,9.00,7.42,111.0,52.42,wait,0,10.0,0.0,Projected inventory (111.0) is above target st...
5,2024-01-02,SKU_C,8.00,5.13,82.0,37.13,wait,0,7.0,0.0,Projected inventory (82.0) is above target sto...
6,2024-01-03,SKU_A,20.20,11.40,138.0,132.60,wait,0,22.0,0.0,Projected inventory (138.0) is above target st...
7,2024-01-03,SKU_B,9.30,2.33,101.0,48.83,wait,0,13.0,0.0,Projected inventory (101.0) is above target st...
8,2024-01-03,SKU_C,7.70,1.81,75.0,32.61,wait,0,9.0,0.0,Projected inventory (75.0) is above target sto...
9,2024-01-04,SKU_A,20.74,8.07,116.0,132.51,order,30,18.0,0.0,Projected inventory (116.0) is below target st...


Baseline daily log sample:


,date,sku,forecast,safety_stock,inventory_position,target_stock,action,order_qty,actual_demand,stockout_units,rationale
0,2024-01-01,SKU_A,18.57,0.0,180.0,92.86,wait,0,19.0,0.0,Projected inventory (180.0) is above target st...
1,2024-01-01,SKU_B,11.79,0.0,120.0,47.14,wait,0,9.0,0.0,Projected inventory (120.0) is above target st...
2,2024-01-01,SKU_C,7.36,0.0,90.0,22.07,wait,0,8.0,0.0,Projected inventory (90.0) is above target sto...
3,2024-01-02,SKU_A,18.57,0.0,161.0,92.86,wait,0,23.0,0.0,Projected inventory (161.0) is above target st...
4,2024-01-02,SKU_B,11.79,0.0,111.0,47.14,wait,0,10.0,0.0,Projected inventory (111.0) is above target st...
5,2024-01-02,SKU_C,7.36,0.0,82.0,22.07,wait,0,7.0,0.0,Projected inventory (82.0) is above target sto...
6,2024-01-03,SKU_A,18.57,0.0,138.0,92.86,wait,0,22.0,0.0,Projected inventory (138.0) is above target st...
7,2024-01-03,SKU_B,11.79,0.0,101.0,47.14,wait,0,13.0,0.0,Projected inventory (101.0) is above target st...
8,2024-01-03,SKU_C,7.36,0.0,75.0,22.07,wait,0,9.0,0.0,Projected inventory (75.0) is above target sto...
9,2024-01-04,SKU_A,18.57,0.0,116.0,92.86,wait,0,18.0,0.0,Projected inventory (116.0) is above target st...


In [5]:
out_dir = Path('inventory_submission_outputs')
out_dir.mkdir(exist_ok=True)
agent_logs.to_csv(out_dir / 'agent_daily_log.csv', index=False)
agent_per_sku.to_csv(out_dir / 'agent_per_sku_kpis.csv', index=False)
agent_summary.to_csv(out_dir / 'agent_summary.csv', index=False)
baseline_logs.to_csv(out_dir / 'baseline_daily_log.csv', index=False)
baseline_per_sku.to_csv(out_dir / 'baseline_per_sku_kpis.csv', index=False)
baseline_summary.to_csv(out_dir / 'baseline_summary.csv', index=False)
comparison.to_csv(out_dir / 'comparison_summary.csv', index=False)
print('Saved output files to', out_dir)

Saved output files to inventory_submission_outputs


## Trade-off reflection
Ordering earlier and carrying more inventory can reduce stockouts and improve fill rate, which supports customer trust and service reliability. However, extra inventory increases holding cost and ties up working capital. This agent tries to balance that trade-off by ordering only when projected inventory falls below lead-time demand plus safety stock.